#                                Ship Classification (PyTorch)




## Amaç
Bu çalışmada gemi görüntülerinden 5 sınıfın (Cargo, Carrier, Cruise, Military, Tanker) sınıflandırılması hedeflenmiştir.

## Deneyler
- Deney 1: NO-AUG (veri artırımı kapalı)
- Deney 2: AUG (veri artırımı açık)

## Çıktılar
- En iyi model checkpoint (best_noaug.pth / best_aug.pth)
- classification report (txt)
- confusion matrix (png)
- koşu karşılaştırması (compare_runs.json)


## Ortam Kurulumu ve Proje Dizinlerinin Tanımlanması

**_Açıklama:_**
Bu hücrede projede kullanılacak temel kütüphaneler içe aktarılmış, Jupyter Notebook içinden çalışıldığı için proje kök dizini belirlenmiş ve src, data, outputs klasörleri tanımlanmıştır. Ayrıca sys.path güncellenerek özel Python modüllerinin sorunsuz import edilmesi sağlanmış ve PyTorch ile CUDA ortam durumu kontrol edilmiştir.

In [19]:
import os, sys, json, random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

import torch

# Proje kökünü ayarla (notebooks içinden çalışıyoruz)
PROJECT_DIR = Path.cwd().parents[0]  # .../GemiSiniflandirmaProjesi
SRC_DIR = PROJECT_DIR / "src"
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "outputs"

sys.path.append(str(SRC_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PROJECT_DIR: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi
SRC_DIR: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\src
DATA_DIR: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\data
OUT_DIR: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs
Torch: 2.9.0+cpu
CUDA available: False


## Deney Parametreleri ve Dosya Yollarının Tanımlanması
**_Açıklama_**:
Bu hücrede deney boyunca kullanılacak temel hiperparametreler (rastgelelik tohumu, giriş görüntü boyutu, batch size, epoch sayısı, öğrenme oranı ve doğrulama oranı) tanımlanmıştır. Ayrıca veri kümesine ait CSV dosyası ve görsellerin bulunduğu dizin yolları belirlenmiş, çıktıların kaydedileceği outputs klasörü güvenli biçimde oluşturulmuştur.

In [20]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
VAL_SIZE = 0.2
LR = 3e-4

NUM_WORKERS = 0
PIN_MEMORY = False

CSV_PATH = DATA_DIR / "train.csv"
IMG_DIR = DATA_DIR / "images"

OUT_DIR.mkdir(parents=True, exist_ok=True)


## Rastgelelik Kontrolü (Seed Ayarı) ve Cihaz Seçimi

**_Açıklama_**:
Bu hücrede deneylerin tekrarlanabilirliğini sağlamak amacıyla Python, NumPy ve PyTorch için rastgelelik tohumları (seed) sabitlenmiştir. Ayrıca CUDA deterministik ayarları yapılandırılarak sonuçların donanıma bağlı değişmesi engellenmiştir. Son olarak, kullanılacak hesaplama cihazı (GPU mevcutsa CUDA, aksi halde CPU) otomatik olarak belirlenmiştir.

In [21]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

## Proje Modüllerinin ve Yardımcı Kütüphanelerin İçe Aktarılması

**_Açıklama_**:
Bu hücrede proje kapsamında geliştirilen özel modüller (ship_dataset.py ve model.py) içe aktarılmaktadır. Veri yolu çözümleme, veri çerçevesi oluşturma, sınıf haritalarının çıkarılması, dönüşümlerin tanımlanması ve DataLoader kurulumu bu modüller aracılığıyla sağlanır. Ayrıca eğitim sürecinde kullanılacak PyTorch, veri bölme, ilerleme çubuğu ve değerlendirme metriklerine ait yardımcı kütüphaneler yüklenmiştir.

In [22]:
from ship_dataset import resolve_paths, load_dataframe, build_class_maps, get_transforms, make_loaders
from model import build_model

# train.py içindeki yardımcı fonksiyonları burada tekrar yazacağız (aynı mantık)
import torch.nn as nn
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix


## Veri Yapılandırması ve Veri Çerçevesinin Oluşturulması

**_Açıklama_**:
Bu hücrede proje dizinleri (data, outputs, images, train.csv) merkezi bir yapılandırma nesnesi (DataConfig) üzerinden tanımlanır. Ardından CSV dosyası okunarak ana veri çerçevesi (DataFrame) oluşturulur. Görseller için tam dosya yolları (filepath) üretilir ve veri setinde eksik ya da bulunamayan görseller olup olmadığı kontrol edilir.

In [23]:
from pathlib import Path
from ship_dataset import DataConfig, load_dataframe

PROJECT_DIR = Path.cwd().parent     # notebooks içindeysen parent proje kökü
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR  = PROJECT_DIR / "outputs"
CSV_PATH = DATA_DIR / "train.csv"
IMG_DIR  = DATA_DIR / "images"

cfg = DataConfig(
    project_dir=PROJECT_DIR,
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    csv_path=CSV_PATH,
    img_dir=IMG_DIR,
)

df = load_dataframe(cfg)

# filepath kolonunu burada biz üretelim (asıl fix)
df["filepath"] = df["image"].apply(lambda x: str(IMG_DIR / x))

print("Toplam kayıt:", len(df))
display(df.head())

missing = df[~df["filepath"].apply(lambda p: Path(p).exists())]
print("Eksik görsel sayısı:", len(missing))


Toplam kayıt: 6252


,image,category,filepath,class_name
0,2823080.jpg,1,C:\Users\berk3\PycharmProjects\GemiSiniflandir...,Cargo
1,2870024.jpg,1,C:\Users\berk3\PycharmProjects\GemiSiniflandir...,Cargo
2,2662125.jpg,2,C:\Users\berk3\PycharmProjects\GemiSiniflandir...,Military
3,2900420.jpg,3,C:\Users\berk3\PycharmProjects\GemiSiniflandir...,Carrier
4,2804883.jpg,2,C:\Users\berk3\PycharmProjects\GemiSiniflandir...,Military


Eksik görsel sayısı: 0


## Veri Bölme, Sınıf Haritaları, Dönüşümler ve DataLoader Kurulumu

**_Açıklama_**:
Bu hücrede veri seti eğitim ve doğrulama kümelerine stratified olarak ayrılır. Sınıf isimleri ile indeksler arasında eşleme (class_to_idx / idx_to_class) oluşturulur. Eğitim ve doğrulama için görüntü dönüşümleri (normalizasyon, yeniden boyutlandırma) tanımlanır. Ardından PyTorch DataLoader nesneleri kurulup tek bir batch alınarak veri şekli ve cihaz uyumluluğu kontrol edilir.

In [24]:
from sklearn.model_selection import train_test_split
from ship_dataset import build_class_maps, get_transforms, make_loaders
import torch

# --- Ayarlar ---
SEED = 42
VAL_SIZE = 0.20
IMG_SIZE = 224
BATCH_SIZE = 16          # senin dediğin gibi 16 yapıyorum
NUM_WORKERS = 0          # Windows/PyCharm için en sorunsuz
PIN_MEMORY = False       # CPU'da uyarı çıkmasın diye False

# --- Split (stratify ile sınıf oranları korunur) ---
train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df["class_name"]
)

print("Train size:", len(train_df), " Val size:", len(val_df))
print("\nTrain dağılımı:")
print(train_df["class_name"].value_counts())
print("\nVal dağılımı:")
print(val_df["class_name"].value_counts())

# --- Sınıf haritaları ---
classes_sorted, class_to_idx, idx_to_class = build_class_maps(df)
num_classes = len(classes_sorted)
print("\nNum classes:", num_classes)
print("Classes:", classes_sorted)

# --- Transformlar (augment kapalı/ açık) ---
train_tfms, val_tfms = get_transforms(
    img_size=IMG_SIZE,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
    augment=False   # şimdilik NO-AUG koşusu
)

# --- DataLoader ---
train_loader, val_loader = make_loaders(
    train_df=train_df,
    val_df=val_df,
    class_to_idx=class_to_idx,
    train_tfms=train_tfms,
    val_tfms=val_tfms,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# --- Hızlı kontrol: 1 batch çek ---
xb, yb = next(iter(train_loader))
print("\nBatch X:", xb.shape, xb.dtype)
print("Batch y:", yb.shape, yb.dtype, "min:", int(yb.min()), "max:", int(yb.max()))
print("Device check (cuda var mı):", torch.cuda.is_available())


Train size: 5001  Val size: 1251

Train dağılımı:
class_name
Cargo       1696
Tanker       973
Military     933
Carrier      733
Cruise       666
Name: count, dtype: int64

Val dağılımı:
class_name
Cargo       424
Tanker      244
Military    234
Carrier     183
Cruise      166
Name: count, dtype: int64

Num classes: 5
Classes: ['Cargo', 'Carrier', 'Cruise', 'Military', 'Tanker']

Batch X: torch.Size([16, 3, 224, 224]) torch.float32
Batch y: torch.Size([16]) torch.int64 min: 0 max: 4
Device check (cuda var mı): False


## Model, Kayıp Fonksiyonu, Optimizasyon ve Öğrenme Oranı Zamanlayıcısı

**_Açıklama_**:
Bu hücrede sınıf sayısına göre model mimarisi oluşturulur ve seçilen cihaza (CPU/GPU) taşınır. Eğitimde kullanılacak kayıp fonksiyonu (CrossEntropyLoss) ve optimizasyon algoritması (Adam) tanımlanır. Doğrulama başarımı artmadığında öğrenme oranını otomatik düşüren ReduceLROnPlateau zamanlayıcısı yapılandırılır ve modelin hangi cihazda hazır olduğu kontrol edilir.

In [28]:
import torch
import torch.nn as nn

from model import build_model

# Model
model = build_model(num_classes=num_classes)   # <-- device parametresini kaldır
model = model.to(device)                       # <-- cihazı burada ayarla

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# (İstersen) LR Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

print("Model hazır. Device:", device)

Model hazır. Device: cpu


## Eğitim ve Doğrulama Döngüleri (Train / Validation Epoch Fonksiyonları)

**_Açıklama_**:
Bu hücrede modelin eğitim ve doğrulama süreçlerini yürüten yardımcı fonksiyonlar tanımlanır.
accuracy_from_logits fonksiyonu model çıktılarından doğruluk hesabı yapar.
train_one_epoch fonksiyonu tek bir epoch boyunca modelin ağırlıklarını güncelleyerek eğitimi gerçekleştirir.
eval_one_epoch fonksiyonu ise gradyan hesaplaması kapalı şekilde modelin doğrulama verisi üzerindeki performansını ölçer.

In [29]:
from tqdm import tqdm
import numpy as np

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for xb, yb in tqdm(loader, desc="train", leave=False):
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc  += accuracy_from_logits(logits, yb)

    return total_loss / len(loader), total_acc / len(loader)

@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    for xb, yb in tqdm(loader, desc="val", leave=False):
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item()
        total_acc  += accuracy_from_logits(logits, yb)

    return total_loss / len(loader), total_acc / len(loader)


## Model Tahmini, Performans Değerlendirme ve Raporlama

**_Açıklama_**:
Bu hücrede eğitilmiş model kullanılarak doğrulama/veri kümesi üzerinde toplu tahmin yapılır ve model performansı detaylı şekilde raporlanır.
predict_all fonksiyonu tüm batch’ler üzerinde modelin tahminlerini toplar.
save_report_and_cm fonksiyonu ise sınıflandırma raporunu (text + JSON) ve karışıklık matrisini (confusion matrix) dosya olarak kaydeder.

In [30]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    for xb, yb in tqdm(loader, desc="predict", leave=False):
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_true.append(yb.numpy())
        y_pred.append(preds)

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return y_true, y_pred

def save_report_and_cm(y_true, y_pred, idx_to_class, out_dir: Path, run_name: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    target_names = [idx_to_class[i] for i in range(len(idx_to_class))]

    # classification report (text + json)
    rep_text = classification_report(y_true, y_pred, target_names=target_names, digits=4)
    (out_dir / f"classification_report_{run_name}.txt").write_text(rep_text, encoding="utf-8")

    rep_dict = classification_report(y_true, y_pred, target_names=target_names, digits=4, output_dict=True)
    with open(out_dir / f"metrics_{run_name}.json", "w", encoding="utf-8") as f:
        json.dump(rep_dict, f, ensure_ascii=False, indent=2)

    # confusion matrix (png)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=target_names, yticklabels=target_names)
    plt.title(f"Confusion Matrix ({run_name})")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(out_dir / f"confusion_matrix_{run_name}.png", dpi=200)
    plt.close()

    print("✅ Kaydedildi:",
          out_dir / f"classification_report_{run_name}.txt",
          out_dir / f"metrics_{run_name}.json",
          out_dir / f"confusion_matrix_{run_name}.png",
          sep="\n- ")

## Eğitim Döngüsü, En İyi Modelin Kaydedilmesi ve Değerlendirme

**_Açıklama_**:
Bu hücre, modelin eğitim sürecinin tamamını yöneten ana fonksiyonu içerir.
run_training fonksiyonu epoch bazlı eğitim ve doğrulama adımlarını yürütür, doğrulama başarımına göre en iyi modeli kaydeder ve eğitim tamamlandıktan sonra bu en iyi modeli geri yükleyerek detaylı performans raporlarını üretir.

In [31]:
def run_training(run_name: str, train_loader, val_loader, augment_flag: bool):
    best_val_acc = -1.0
    best_path = OUT_DIR / f"best_{run_name}.pth"

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = eval_one_epoch(model, val_loader, criterion, device)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        scheduler.step(va_acc)

        print(f"[{run_name}] Epoch {epoch}/{EPOCHS} | "
              f"train_acc={tr_acc:.4f} val_acc={va_acc:.4f} "
              f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save({"model_state_dict": model.state_dict(),
                        "class_to_idx": class_to_idx,
                        "idx_to_class": idx_to_class,
                        "img_size": IMG_SIZE,
                        "run_name": run_name}, best_path)
            print(f"✅ Best saved: {best_path} (val_acc={best_val_acc:.4f})")

    # best modeli geri yükle, rapor üret
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    y_true, y_pred = predict_all(model, val_loader, device)
    save_report_and_cm(y_true, y_pred, idx_to_class, OUT_DIR, run_name)

    return history, best_val_acc, best_path


## Deney 1: Veri Artırımı Olmadan Eğitim (NO-AUG)

**_Açıklama_**:
Bu adımda model, herhangi bir veri artırımı (augmentation) uygulanmadan eğitilmiştir. Amaç, veri artırımı kullanılmadığında modelin doğrulama performansını ölçmek ve daha sonra veri artırımlı (AUG) eğitimle adil bir karşılaştırma yapabilmektir. Eğitim sürecinde en iyi doğrulama başarımına sahip model otomatik olarak kaydedilmiş ve detaylı performans raporları üretilmiştir.

In [32]:
# NO-AUG için loader'ların zaten var: train_loader, val_loader
history_noaug, best_noaug, path_noaug = run_training(
    run_name="noaug",
    train_loader=train_loader,
    val_loader=val_loader,
    augment_flag=False
)

print("NO-AUG best val acc:", best_noaug)


[noaug] Epoch 1/10 | train_acc=0.8333 val_acc=0.9190 train_loss=0.4580 val_loss=0.1927
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_noaug.pth (val_acc=0.9190)


[noaug] Epoch 2/10 | train_acc=0.9347 val_acc=0.9248 train_loss=0.1825 val_loss=0.1925
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_noaug.pth (val_acc=0.9248)


[noaug] Epoch 3/10 | train_acc=0.9665 val_acc=0.9351 train_loss=0.1051 val_loss=0.1872
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_noaug.pth (val_acc=0.9351)


[noaug] Epoch 4/10 | train_acc=0.9782 val_acc=0.9399 train_loss=0.0654 val_loss=0.2073
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_noaug.pth (val_acc=0.9399)


[noaug] Epoch 5/10 | train_acc=0.9774 val_acc=0.9494 train_loss=0.0633 val_loss=0.1437
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_noaug.pth (val_acc=0.9494)


...

[noaug] Epoch 7/10 | train_acc=0.9798 val_acc=0.9248 train_loss=0.0597 val_loss=0.2434


[noaug] Epoch 8/10 | train_acc=0.9876 val_acc=0.9320 train_loss=0.0444 val_loss=0.2223


[noaug] Epoch 9/10 | train_acc=0.9936 val_acc=0.9549 train_loss=0.0231 val_loss=0.1380
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_noaug.pth (val_acc=0.9549)


[noaug] Epoch 10/10 | train_acc=0.9974 val_acc=0.9541 train_loss=0.0085 val_loss=0.1565


✅ Kaydedildi:
- C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\classification_report_noaug.txt
- C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\metrics_noaug.json
- C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\confusion_matrix_noaug.png
NO-AUG best val acc: 0.9549050632911392


## Deney 2: Veri Artırımı ile Eğitim (AUG)

**_Açıklama_**:
Bu adımda eğitim verilerine rastgele veri artırımı (augmentation) uygulanarak model yeniden eğitilmiştir. Amaç, veri artırımının modelin genelleme yeteneği üzerindeki etkisini incelemek ve NO-AUG deney sonuçlarıyla karşılaştırmaktır. Eğitim ve doğrulama veri yükleyicileri artırımlı dönüşümlerle yeniden oluşturulmuş, model sıfırdan başlatılarak adil bir karşılaştırma sağlanmıştır. En iyi doğrulama başarımına sahip model kaydedilmiş ve performans metrikleri raporlanmıştır.

In [34]:
# AUG transform
train_tfms_aug, val_tfms_aug = get_transforms(
    img_size=IMG_SIZE,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
    augment=True
)

train_loader_aug, val_loader_aug = make_loaders(
    train_df=train_df,
    val_df=val_df,
    class_to_idx=class_to_idx,
    train_tfms=train_tfms_aug,
    val_tfms=val_tfms_aug,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# Modeli AUG için sıfırdan başlatmak daha adil karşılaştırma
model = build_model(num_classes=num_classes)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

history_aug, best_aug, path_aug = run_training(
    run_name="aug",
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    augment_flag=True
)

print("AUG best val acc:", best_aug)


[aug] Epoch 1/10 | train_acc=0.8329 val_acc=0.9375 train_loss=0.4758 val_loss=0.1862
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_aug.pth (val_acc=0.9375)


[aug] Epoch 2/10 | train_acc=0.9155 val_acc=0.9438 train_loss=0.2282 val_loss=0.1586
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_aug.pth (val_acc=0.9438)


[aug] Epoch 3/10 | train_acc=0.9537 val_acc=0.9312 train_loss=0.1389 val_loss=0.1802


[aug] Epoch 4/10 | train_acc=0.9591 val_acc=0.9391 train_loss=0.1087 val_loss=0.1983


[aug] Epoch 5/10 | train_acc=0.9667 val_acc=0.9509 train_loss=0.0913 val_loss=0.1846
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_aug.pth (val_acc=0.9509)


...

[aug] Epoch 7/10 | train_acc=0.9784 val_acc=0.9494 train_loss=0.0604 val_loss=0.1707


[aug] Epoch 8/10 | train_acc=0.9818 val_acc=0.9517 train_loss=0.0618 val_loss=0.1460
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_aug.pth (val_acc=0.9517)


[aug] Epoch 9/10 | train_acc=0.9772 val_acc=0.9533 train_loss=0.0666 val_loss=0.1634
✅ Best saved: C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\best_aug.pth (val_acc=0.9533)


[aug] Epoch 10/10 | train_acc=0.9848 val_acc=0.9446 train_loss=0.0441 val_loss=0.1863


✅ Kaydedildi:
- C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\classification_report_aug.txt
- C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\metrics_aug.json
- C:\Users\berk3\PycharmProjects\GemiSiniflandirmaProjesi\outputs\confusion_matrix_aug.png
AUG best val acc: 0.9533227848101266


## Deney Sonuçlarının Karşılaştırılması ve Kaydedilmesi

**_Açıklama_**:
Bu adımda veri artırımı olmadan (NO-AUG) ve veri artırımı ile (AUG) gerçekleştirilen iki deneyin en iyi doğrulama doğrulukları (best_val_acc) ve ilgili en iyi model dosya yolları karşılaştırılmıştır. Ayrıca deneylerde kullanılan temel hiperparametreler (batch size, epoch sayısı, görüntü boyutu, öğrenme oranı ve doğrulama oranı) tek bir yapı altında toplanmıştır. Elde edilen karşılaştırma sonuçları daha sonra analiz ve raporlama amacıyla compare_runs.json dosyasına kaydedilmiştir.

In [35]:
compare = {
    "noaug": {"best_val_acc": float(best_noaug), "best_path": str(path_noaug)},
    "aug":   {"best_val_acc": float(best_aug),   "best_path": str(path_aug)},
    "settings": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "img_size": IMG_SIZE,
        "lr": LR,
        "val_size": VAL_SIZE
    }
}
with open(OUT_DIR / "compare_runs.json", "w", encoding="utf-8") as f:
    json.dump(compare, f, ensure_ascii=False, indent=2)

compare


{'noaug': {'best_val_acc': 0.9549050632911392,
  'best_path': 'C:\\Users\\berk3\\PycharmProjects\\GemiSiniflandirmaProjesi\\outputs\\best_noaug.pth'},
 'aug': {'best_val_acc': 0.9533227848101266,
  'best_path': 'C:\\Users\\berk3\\PycharmProjects\\GemiSiniflandirmaProjesi\\outputs\\best_aug.pth'},
 'settings': {'batch_size': 16,
  'epochs': 10,
  'img_size': 224,
  'lr': 0.0003,
  'val_size': 0.2}}